In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all of the words
words = open('../names.txt', 'r').read().splitlines()
print('first few:',words[:8])
print('longest word:', max(len(w) for w in words))
print('total words:', len(words))

first few: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
longest word: 15
total words: 32033


In [3]:
# build the vocabulary of characters and mappings to/from integers
import string
START_STOP_TOKEN = '.'
tokens = [START_STOP_TOKEN, *string.ascii_lowercase]
stoi = { s: i for i, s in enumerate(tokens) }
itos = { i: s for s, i in stoi.items() }
N_TOKENS = len(tokens)
print(itos)
print('count of tokens:', N_TOKENS)

{0: '.', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z'}
count of tokens: 27


In [4]:
# build the dataset

BLOCK_SIZE = 3 # context length; number of chars used to predict the next ones
SPLIT_1 = 0.8 # 80% train
SPLIT_2 = 0.9 # 10% dev, 10% test

def build_dataset(words):
    X, Y = [], []
    for w in words:
        #print(); print(w)
        context = [0] * BLOCK_SIZE
        for ch in w + START_STOP_TOKEN:
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '-->', itos[ix])
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

# -- with train, dev, test split --
import random
random.seed(42)
shuffled_words = words[:]
random.shuffle(shuffled_words)

n1 = int(SPLIT_1 * len(shuffled_words))
n2 = int(SPLIT_2 * len(shuffled_words))

print('trainset:', end=' ')
Xtr, Ytr = build_dataset(shuffled_words[:n1])
print('devset  :', end=' ')
Xdev, Ydev = build_dataset(shuffled_words[n1:n2])
print('testset :', end=' ')
Xte, Yte = build_dataset(shuffled_words[n2:])

trainset: torch.Size([182625, 3]) torch.Size([182625])
devset  : torch.Size([22655, 3]) torch.Size([22655])
testset : torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# -- All the blocks so far were unchanged from the previous lecture ---
# Now time for new stuff

In [6]:
# helper function for double-checking manual gradients against PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact {str(ex):5s} | approx: {str(app):5s} | maxdiff: {maxdiff}')

In [7]:
N_EMBED = 10 # dimensionality of the char embedding vectors
INPUT_SIZE = BLOCK_SIZE * N_EMBED
N_HIDDEN = 64 # number of neurons in hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)
randn = lambda *size: torch.randn(size, generator=g)
kaiming_scale = lambda fan_in: (5/3)/(fan_in**0.5)

# Note: We are doing some non-standard initializations, as sometimes initializing
# with e.g. all zeros could mask an incorrect backprop implementation.

C =  randn(N_TOKENS, N_EMBED)
# -- Layer 1 --
W1 = randn(INPUT_SIZE, N_HIDDEN) * kaiming_scale(INPUT_SIZE)
b1 = randn(N_HIDDEN)             * 0.1 # keep b1 so we can check its grad, even tho batch norm makes it useless
# -- Layer 2 --
W2 = randn(N_HIDDEN, N_TOKENS)   * 0.1
b2 = randn(N_TOKENS)             * 0.1

# BatchNorm params
bn_gain = randn(1, N_HIDDEN) * 0.1 + 1.0
bn_bias = randn(1, N_HIDDEN) * .01

bn_mean_running = torch.zeros((1, N_HIDDEN))
bn_std_running = torch.ones((1, N_HIDDEN))
 
parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
for p in parameters:
    p.requires_grad = True

print('num params:', sum(p.nelement() for p in parameters))

num params: 4137


In [8]:
BATCH_SIZE = 32
n = BATCH_SIZE # for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (BATCH_SIZE,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

In [12]:
# forward pass, "chunked" into smaller steps that are easier to backward one at a time

emb = C[Xb] # embed characters into vectors
embcat = emb.view(emb.shape[0], -1) # concat the vectors

# Linear Layer 1
h_pre_bn = embcat @ W1 + b1 # hidden layer pre-activation

# BatchNorm Layer
bn_meani = 1/n * h_pre_bn.sum(0, keepdim=True)
bn_diff = h_pre_bn - bn_meani
bn_diff2 = bn_diff**2
bn_var = 1/(n-1) * bn_diff2.sum(0, keepdim=True) # Note: Bessel's correction (dividing by n-1, not n)
bn_var_inv = (bn_var + 1e-5)**(-0.5)
bn_raw = bn_diff * bn_var_inv
h_pre_act = bn_gain * bn_raw + bn_bias

# Non-linearity
h = torch.tanh(h_pre_act) # hidden layer

# Linear Layer 2
logits = h @ W2 + b2

# Cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum ** -1 # If we use (1.0 / counts_sum) instead then we can't get backprop to be exact...
probs = counts * counts_sum_inv
log_probs = probs.log()
loss = -log_probs[range(n), Yb].mean()

for p in parameters:
    p.grad = None
for t in [log_probs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, h_pre_act, bn_raw,
          bn_var_inv, bn_var, bn_diff, bn_diff2, h_pre_bn, bn_meani, embcat, emb]:
    t.retain_grad()

loss.backward()
loss

tensor(3.5556, grad_fn=<NegBackward0>)

In [49]:
Xb[:5]

tensor([[ 0, 11, 25],
        [ 9, 18, 15],
        [ 0,  0,  0],
        [ 1, 12,  9],
        [ 1, 14, 14]])

In [128]:
def inverse_index_add_3d(a, b, x_shape):
    # Create a tensor of zeros with the same shape as x
    y = torch.zeros(x_shape).long()
    # print(a.view(a.shape[0], -1).shape, b.shape, x_shape)
    
    # Perform scatter_add along the first dimension
    y.scatter_add_(0, b, a.view(a.shape[0], -1))
    return y

def inverse_index_add_3d_v2(a, b, x_shape):
    y = torch.zeros(x_shape).long()

    for k in range(b.shape[0]):
        for j in range(b.shape[1]):
            i = b[k][j]
            y[i] += a[k][j]
    return y

def inverse_index_add_3d_v3(a, b, x):
    output = torch.zeros_like(x)
    for i in range(a.size(1)):  # Iterate over the second dimension of a
        output.scatter_add_(0, b[:, i:i+1], a[:, i, :])
    return output

def inverse_index_add_3d_v3(a, b, x):
    output = torch.zeros_like(x)
    for i in range(a.size(1)):  # Iterate over the second dimension of a
        output.scatter_add_(0, b[:, i:i+1].expand(-1, a.size(-1)), a[:, i, :])
    return output

def inverse_index_add_3d_v4(a, b, x):
    print("~~~~~~~~~~~~~~~~~~")
    print(b.view(-1))
    print(a.view(-1, a.size(-1)))
    print("~~~~~~~~~~~~~~~~~~")
    return torch.zeros_like(x).index_add(
        0,
        b.view(-1),
        a.view(-1, a.size(-1)),
    )

# Example usage
x = torch.randint(20, (2, 2))  # Original 2D tensor
b = torch.randint(0, 2, (4, 2))  # 2D index tensor
a = x[b]  # This will be 3D: (4, 2, 3)

print("Original x:", x)
print("Index b:", b)
print("Selected a:", a)
print(a.shape, b.shape, x.shape)

y = inverse_index_add_3d(a, b, x.shape)
y2 = inverse_index_add_3d_v2(a, b, x.shape)
y3 = inverse_index_add_3d_v3(a, b, x)
y4 = inverse_index_add_3d_v4(a, b, x)

print("Reconstructed y:", y)
print("Reconstructed y2:", y2)
print("Reconstructed y3:", y3)
print("Reconstructed y4:", y4)

Original x: tensor([[17,  7],
        [13,  3]])
Index b: tensor([[0, 0],
        [1, 0],
        [0, 1],
        [0, 1]])
Selected a: tensor([[[17,  7],
         [17,  7]],

        [[13,  3],
         [17,  7]],

        [[17,  7],
         [13,  3]],

        [[17,  7],
         [13,  3]]])
torch.Size([4, 2, 2]) torch.Size([4, 2]) torch.Size([2, 2])
~~~~~~~~~~~~~~~~~~
tensor([0, 0, 1, 0, 0, 1, 0, 1])
tensor([[17,  7],
        [17,  7],
        [13,  3],
        [17,  7],
        [17,  7],
        [13,  3],
        [17,  7],
        [13,  3]])
~~~~~~~~~~~~~~~~~~
Reconstructed y: tensor([[51, 10],
        [13, 14]])
Reconstructed y2: tensor([[85, 35],
        [39,  9]])
Reconstructed y3: tensor([[85, 35],
        [39,  9]])
Reconstructed y4: tensor([[85, 35],
        [39,  9]])


In [105]:
# emb = C[Xb]
print('emb = C[Xb]')
print('------------------------------')
print(bn_bias.shape)
print('emb   :', emb.shape)
print('d_emb :', d_emb.shape)
# print('Xb    :', Xb.shape)
print('C     :', C.shape)
print('------------------------------')
print('Xb    :', Xb.shape)
print('------------------------------')
print('embcat:', embcat.shape)


# (C)      a = [1,4,9,16,25,36,49,68]
# (Xb)     b = [1,1,5,2,6,5]
# (emb) a[b] = [1,1,25,4,36,25]

# d_emb = [de1, de2, de3, de4, de5, de6]
# dC

# dC = torch.zeros(C.shape[0])
# dC[Xb] = 1
# 

print()
print('-----------')
a = torch.tensor([0,1,4,9,16,25,36,49,64])
b = torch.tensor([1,1,5,2,6,5])

print(a)
print(b)
print(a[b])
print('-----------')

zz = torch.zeros_like(a).scatter_add(0, b, torch.ones_like(a))
print(zz)

max = a.shape[0]
b_counts = torch.histc(b.flatten().float(), bins=max, max=max).long()
print(b_counts)

emb = C[Xb]
------------------------------
torch.Size([1, 64])
emb   : torch.Size([32, 3, 10])
d_emb : torch.Size([32, 3, 10])
C     : torch.Size([27, 10])
------------------------------
Xb    : torch.Size([32, 3])
------------------------------
embcat: torch.Size([32, 30])

-----------
tensor([ 0,  1,  4,  9, 16, 25, 36, 49, 64])
tensor([1, 1, 5, 2, 6, 5])
tensor([ 1,  1, 25,  4, 36, 25])
-----------
tensor([0, 2, 1, 0, 0, 2, 1, 0, 0])
tensor([0, 2, 1, 0, 0, 2, 1, 0, 0])


In [129]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)

d_log_probs = torch.zeros_like(log_probs)
d_log_probs[range(n), Yb] = -1.0 / n
d_probs = (1 / probs) * d_log_probs
d_counts_sum_inv = (counts * d_probs).sum(1, keepdim=True) # we sum because of the broadcasting
d_counts = counts_sum_inv * d_probs
d_counts_sum = (-1 * (counts_sum ** -2)) * d_counts_sum_inv
d_counts += torch.ones_like(counts) * d_counts_sum
d_norm_logits = norm_logits.exp() * d_counts
d_logits = d_norm_logits.clone()
d_logit_maxes = (-1 * d_norm_logits).sum(1, keepdim=True) # sum for broadcasting / shape
d_logits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * d_logit_maxes

dh = d_logits @ W2.T
dW2 = h.T @ d_logits
db2 = d_logits.sum(0)

dh_pre_act = (1 - h**2) * dh # d/dx tanh(x) = 1 - tanh(x)^2
d_bn_gain = (bn_raw * dh_pre_act).sum(0, keepdims=True)
d_bn_bias = dh_pre_act.sum(0, keepdims=True)
d_bn_raw = bn_gain * dh_pre_act

d_bn_var_inv = (bn_diff * d_bn_raw).sum(0, keepdims=True)
d_bn_diff = bn_var_inv * d_bn_raw
d_bn_var = (-0.5) * (bn_var + 1e-5)**(-1.5) * d_bn_var_inv

d_bn_diff2 = 1.0 / (n-1) * torch.ones_like(bn_diff2) * d_bn_var
d_bn_diff += (2*bn_diff) * d_bn_diff2
d_bn_meani = (-1 * d_bn_diff).sum(0, keepdims=True)

dh_pre_bn = d_bn_diff.clone()
dh_pre_bn += (1.0/n) * torch.ones_like(h_pre_bn) * d_bn_meani

# -----------------

cmp('logprobs', d_log_probs, log_probs)
cmp('probs', d_probs, probs)
cmp('counts_sum_inv', d_counts_sum_inv, counts_sum_inv)
cmp('counts_sum', d_counts_sum, counts_sum)
cmp('counts', d_counts, counts)
cmp('norm_logits', d_norm_logits, norm_logits)
cmp('logit_maxes', d_logit_maxes, logit_maxes)
cmp('logits', d_logits, logits)

cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

cmp('h_pre_act', dh_pre_act, h_pre_act)
cmp('bn_gain', d_bn_gain, bn_gain)
cmp('bn_bias', d_bn_bias, bn_bias)
cmp('bnraw', d_bn_raw, bn_raw)

cmp('bnvar_inv', d_bn_var_inv, bn_var_inv)
cmp('bn_var', d_bn_var, bn_var)
cmp('bn_diff2', d_bn_diff2, bn_diff2)
cmp('bn_diff', d_bn_diff, bn_diff)
cmp('bn_meani', d_bn_meani, bn_meani)

cmp('h_pre_bn', dh_pre_bn, h_pre_bn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

logprobs        | exact True  | approx: True  | maxdiff: 0.0
probs           | exact True  | approx: True  | maxdiff: 0.0
counts_sum_inv  | exact True  | approx: True  | maxdiff: 0.0
counts_sum      | exact True  | approx: True  | maxdiff: 0.0
counts          | exact True  | approx: True  | maxdiff: 0.0
norm_logits     | exact True  | approx: True  | maxdiff: 0.0
logit_maxes     | exact True  | approx: True  | maxdiff: 0.0
logits          | exact True  | approx: True  | maxdiff: 0.0
h               | exact True  | approx: True  | maxdiff: 0.0
W2              | exact True  | approx: True  | maxdiff: 0.0
b2              | exact True  | approx: True  | maxdiff: 0.0
h_pre_act       | exact True  | approx: True  | maxdiff: 0.0
bn_gain         | exact True  | approx: True  | maxdiff: 0.0
bn_bias         | exact True  | approx: True  | maxdiff: 0.0
bnraw           | exact True  | approx: True  | maxdiff: 0.0
bnvar_inv       | exact True  | approx: True  | maxdiff: 0.0
bn_var          | exact 

In [ ]:
# z = x @ y + b
# x.shape = (2,4)
# y.shape = (4,3)
# b.shape = (3)
# z.shape = (2,3)

# l11 = h11*w11 + h12*w21 + h13*w31 + h14*w41 + b1
# l12 = h11*w12 + h12*w22 + h13*w32 + h14*w42 + b2
# l13 = h11*w13 + ...                         + b3

# l21 = h21*w11 + h21*w12 + ...               + b1
# ...

# dL/b1 = (dL/dl11 * 1) + (dL/dl21 * 1)

In [ ]:
# l = h @ w
# h.shape = (2,4)
# w.shape = (4,3)
# l.shape = (2,3)


# l11 = h11*w11 + h12*w21 + h13*w31 + h14*w41
# l12 = h11*w12 + h12*w22 + h13*w32 + h14*w42
# l13 = h11*w13 + ...

# l21 = h21*w11 + h21*w12 + ...

# dL/dh11 = (dL/dl11 * w11) + (dL/dl12 * w12) + (dL/dl13 * w13)
# dL/dh12 = (dL/dl11 * w21) + (dL/dl12 * w22) + (dL/dl13 * w23)
# dL/dh13 = (dL/dl11 * w31) + (dL/dl12 * w32) + (dL/dl13 * w33)
# dL/dh14 = (dL/dl11 * w41) + (dL/dl12 * w42) + (dL/dl13 * w43)

# dL/dh21 = (dL/dl21 * w11) + (dL/dl22 * w12) + (dL/dl23 * w13)
# dL/dh22 = (dL/dl21 * w21) + (dL/dl22 * w22) + (dL/dl23 * w23)
# dL/dh23 = (dL/dl21 * w31) + (dL/dl22 * w32) + (dL/dl23 * w33)
# dL/dh24 = (dL/dl21 * w41) + (dL/dl22 * w42) + (dL/dl23 * w43)

# dh.shape = (2,4)
